# Google Cloud Dataflow Pipeline with Apache Beam & GCS

This project is wrapped and managed using **`uv`** (Python 3.12, keyless ADC authentication, GCP Free Tier compatible).

### Quick Start with `uv`:
- **Kernel**: Select the kernel **`Python (uv: GCP Dataflow)`** or choose the interpreter at `.venv/Scripts/python.exe`.
- **Run via CLI**:
  ```bash
  uv run python -m gcp.pipeline
  ```
- **Add new dependencies**:
  ```bash
  uv add <package-name>
  ```

### Keyless Authentication (Free Tier):
No service account JSON key files required. Authenticate using your user credentials via `gcloud`:
```bash
gcloud auth login
gcloud auth application-default login
gcloud config set project <YOUR_GCP_PROJECT_ID>
gcloud auth application-default set-quota-project <YOUR_GCP_PROJECT_ID>
```

### Pipeline Architecture:
1. **Input**: `Employers_data.csv` stored in Google Cloud Storage (`gs://<YOUR_BUCKET_NAME>/input/Employers_data.csv`)
2. **Processing (Apache Beam)**:
   - Read CSV lines and skip header
   - Parse and cast columns (`Employee_ID`, `Name`, `Age`, `Department`, `Salary`, etc.)
   - Extract Key-Value pairs: `(Department, Salary)`
   - Aggregate metrics: compute **Average Salary per Department**
   - Format results into CSV format
3. **Output Sink**: Partitioned output files written back to GCS (`gs://<YOUR_BUCKET_NAME>/output/department_salary_summary-*.csv`)
4. **Execution Runners**:
   - **DirectRunner**: For local testing and validation
   - **DataflowRunner**: For fully managed, distributed execution on Google Cloud Platform

In [1]:
# Cell 1: Environment check (managed by uv)
# Note: When running in VS Code or Jupyter, select the kernel: 'Python (uv: GCP Dataflow)'
# If running in an external notebook like Google Colab, uncomment the line below:
# !pip install --quiet "apache-beam[gcp]" google-cloud-storage google-auth pandas

import sys
print(f"Python Executable: {sys.executable}")
print("Project dependencies are managed by uv (.venv).")

Python Executable: C:\Users\debar\AppData\Local\Python\pythoncore-3.14-64\python.exe
Project dependencies are managed by uv (.venv).


In [2]:
# Cell 2: Import required modules and libraries
import os
import csv
import json
import logging
import glob
import google.auth
from google.cloud import storage
import apache_beam as beam
from apache_beam.options.pipeline_options import (
    PipelineOptions,
    GoogleCloudOptions,
    StandardOptions,
    SetupOptions,
    WorkerOptions
)
import apache_beam.transforms.combiners as combiners

logging.basicConfig(level=logging.INFO)
print(f"Apache Beam version: {beam.__version__}")

Apache Beam version: 2.76.0


### Cell 3: Configure GCP Project & Verify Authentication (Keyless ADC)
Since you are using the GCP Free Tier, no service account key file is required. The Google Cloud client libraries automatically use your **Application Default Credentials (ADC)**.

In [3]:
# Cell 3: GCP Configuration (Keyless / Free Tier)
# =====================================================================
# UPDATE THE PLACEHOLDER VALUES BELOW WITH YOUR GCP PROJECT DETAILS
# =====================================================================

# 1. Google Cloud Project ID
PROJECT_ID = "debaranjan-practice"  # e.g., "my-cloud-project-12345"

# 2. Google Cloud Storage Bucket Name (omit 'gs://')
BUCKET_NAME = "my-firsct-bucket-debproj"  # e.g., "my-company-dataflow-bucket"

# 3. Compute Region for Dataflow execution (us-central1 is standard for free tier)
REGION = "asia-south1"  # e.g., "us-central1", "us-east1", "europe-west1"

# 4. Check and verify Application Default Credentials (ADC)
try:
    credentials, detected_project = google.auth.default()
    print("SUCCESS: Authenticated via Application Default Credentials (ADC).")
    if detected_project:
        print(f"Default detected project: {detected_project}")
except Exception as e:
    print(f"ADC Notice: {e}")
    print("To authenticate without a key, run in your terminal:")
    print("  gcloud auth application-default login")

# 5. Cloud Storage Paths for Dataflow
INPUT_GCS_PATH = f"gs://{BUCKET_NAME}/input/Employers_data.csv"
OUTPUT_GCS_PATH = f"gs://{BUCKET_NAME}/output/department_salary_summary"
STAGING_LOCATION = f"gs://{BUCKET_NAME}/staging"
TEMP_LOCATION = f"gs://{BUCKET_NAME}/temp"
JOB_NAME = "employee-salary-dataflow-job"

print("\n--- GCP Pipeline Configuration ---")
print(f"Project ID:       {PROJECT_ID}")
print(f"Bucket Name:      {BUCKET_NAME}")
print(f"Region:           {REGION}")
print(f"Input GCS:        {INPUT_GCS_PATH}")
print(f"Output GCS:       {OUTPUT_GCS_PATH}")
print(f"Staging Location: {STAGING_LOCATION}")
print(f"Temp Location:    {TEMP_LOCATION}")

SUCCESS: Authenticated via Application Default Credentials (ADC).
Default detected project: debaranjan-practice

--- GCP Pipeline Configuration ---
Project ID:       debaranjan-practice
Bucket Name:      my-firsct-bucket-debproj
Region:           asia-south1
Input GCS:        gs://my-firsct-bucket-debproj/input/Employers_data.csv
Output GCS:       gs://my-firsct-bucket-debproj/output/department_salary_summary
Staging Location: gs://my-firsct-bucket-debproj/staging
Temp Location:    gs://my-firsct-bucket-debproj/temp


In [4]:
# Cell 4: Upload Local CSV to Google Cloud Storage (Utility)
# Helper function to upload 'SourceFiles/Employers_data.csv' to your GCS bucket
# (Uses ADC - no service account key needed)

def upload_local_file_to_gcs(local_path: str, bucket_name: str, target_blob_name: str):
    """Uploads a local file to a GCS bucket using user ADC credentials."""
    try:
        client = storage.Client(project=PROJECT_ID)
        bucket = client.bucket(bucket_name)
        blob = bucket.blob(target_blob_name)
        blob.upload_from_filename(local_path)
        print(f"Uploaded '{local_path}' to 'gs://{bucket_name}/{target_blob_name}' successfully.")
    except Exception as err:
        print(f"Upload error: {err}")

# Locate local Employers_data.csv
local_candidates = ["../SourceFiles/Employers_data.csv", "SourceFiles/Employers_data.csv"]
local_csv_path = next((p for p in local_candidates if os.path.exists(p)), None)

if local_csv_path:
    print(f"Found source file: {local_csv_path}")
    # Uncomment the line below once BUCKET_NAME is created and ADC is logged in:
    upload_local_file_to_gcs(local_csv_path, BUCKET_NAME, "input/Employers_data.csv")
else:
    print("Could not locate 'Employers_data.csv'. Please check the path.")

Found source file: ../SourceFiles/Employers_data.csv
Uploaded '../SourceFiles/Employers_data.csv' to 'gs://my-firsct-bucket-debproj/input/Employers_data.csv' successfully.


In [5]:
# Cell 5: Define Apache Beam Transformation Classes (DoFn)

class ParseAndCleanEmployeeDoFn(beam.DoFn):
    """
    Parses CSV lines, skips header row, handles basic cleansing, and casts types.
    Fields: Employee_ID,Name,Age,Gender,Department,Job_Title,Experience_Years,Education_Level,Location,Salary
    """
    def process(self, line: str):
        line = line.strip()
        if not line or line.startswith("Employee_ID"):
            return
        
        reader = csv.reader([line])
        for row in reader:
            if len(row) >= 10:
                try:
                    yield {
                        "employee_id": int(row[0].strip()),
                        "name": row[1].strip(),
                        "age": int(row[2].strip()),
                        "gender": row[3].strip(),
                        "department": row[4].strip(),
                        "job_title": row[5].strip(),
                        "experience_years": int(row[6].strip()),
                        "education_level": row[7].strip(),
                        "location": row[8].strip(),
                        "salary": float(row[9].strip()),
                    }
                except (ValueError, IndexError):
                    # Skip rows with malformed numeric values
                    continue


class ExtractDepartmentSalaryDoFn(beam.DoFn):
    """Extracts (department, salary) tuples for key-based aggregation."""
    def process(self, record: dict):
        yield (record["department"], record["salary"])


class FormatSummaryStatsDoFn(beam.DoFn):
    """Formats (department, average_salary) tuple into a CSV line."""
    def process(self, element):
        department, avg_salary = element
        yield f"{department},{avg_salary:.2f}"

In [7]:
# Cell 6: Test Apache Beam Pipeline Locally (DirectRunner)
# Run this locally before launching on GCP Dataflow to verify transformations
# (Requires NO GCP connection - 100% free and offline)

local_test_file = next((p for p in ["../SourceFiles/Employers_data.csv", "SourceFiles/Employers_data.csv"] if os.path.exists(p)), None)
local_out_dir = "./local_output"
os.makedirs(local_out_dir, exist_ok=True)
local_output_prefix = os.path.join(local_out_dir, "dept_salary_summary")

print("Executing local pipeline with DirectRunner...")
with beam.Pipeline(runner="DirectRunner") as p:
    (
        p
        | "ReadLocalCSV" >> beam.io.ReadFromText(local_test_file)
        | "ParseCleanRecords" >> beam.ParDo(ParseAndCleanEmployeeDoFn())
        | "ExtractDeptSalary" >> beam.ParDo(ExtractDepartmentSalaryDoFn())
        | "ComputeAvgSalary" >> combiners.Mean.PerKey()
        | "FormatResults" >> beam.ParDo(FormatSummaryStatsDoFn())
        | "WriteLocalOutput" >> beam.io.WriteToText(
            file_path_prefix=local_output_prefix,
            file_name_suffix=".csv",
            header="Department,Average_Salary"
        )
    )

print("Local test run finished! Checking output:")
out_files = glob.glob(f"{local_output_prefix}*")
for fpath in out_files:
    print(f"\n--- {fpath} ---")
    with open(fpath, "r") as f:
        print(f.read().strip())

INFO:apache_beam.io.iobase:*** WriteImpl min_shards undef so it's 1, and we write per Bundle
INFO:apache_beam.runners.worker.statecache:Creating state cache with size 104857600


Executing local pipeline with DirectRunner...


INFO:apache_beam.io.filebasedsink:Renamed 1 shards in 0.00 seconds.


Local test run finished! Checking output:

--- ./local_output\dept_salary_summary-00000-of-00001.csv ---
Department,Average_Salary
Engineering,90680.33
Sales,127309.77
Finance,130376.18
HR,126400.60
Marketing,101734.57
Product,116676.33


### Cell 7: Configure Dataflow Pipeline Options (Free Tier Friendly)

When submitting to Dataflow:
1. Your local client uses **ADC** to submit the job.
2. The Dataflow worker VMs in GCP automatically run under your project's **Compute Engine default service account** (`<PROJECT_NUMBER>-compute@developer.gserviceaccount.com`).

> **One-time IAM setup (if needed)**:
> Ensure the default Compute Engine service account has the `Dataflow Worker` and `Storage Object Admin` roles:
> ```bash
> PROJECT_NUMBER=$(gcloud projects describe YOUR_GCP_PROJECT_ID --format="value(projectNumber)")
> gcloud projects add-iam-policy-binding YOUR_GCP_PROJECT_ID \
>     --member="serviceAccount:${PROJECT_NUMBER}-compute@developer.gserviceaccount.com" \
>     --role="roles/dataflow.worker"
> gcloud projects add-iam-policy-binding YOUR_GCP_PROJECT_ID \
>     --member="serviceAccount:${PROJECT_NUMBER}-compute@developer.gserviceaccount.com" \
>     --role="roles/storage.objectAdmin"
> ```

In [11]:
# Cell 7: Configure GCP Dataflow Pipeline Options

dataflow_pipeline_args = [
    f"--project={PROJECT_ID}",
    f"--region={REGION}",
    "--runner=DataflowRunner",
    f"--staging_location={STAGING_LOCATION}",
    f"--temp_location={TEMP_LOCATION}",
    f"--job_name={JOB_NAME}",
    "--save_main_session",  # Essential for serialization of functions & classes in __main__
    # Cost-effective worker settings for Free Tier / small batches:
    "--worker_machine_type=e2-standard-2",
    "--num_workers=1",
    "--max_num_workers=2"
]

dataflow_options = PipelineOptions(dataflow_pipeline_args)
print("Dataflow pipeline options configured successfully for Free Tier.")

Dataflow pipeline options configured successfully for Free Tier.


In [14]:
# Cell 8: Submit & Execute Apache Beam Pipeline on GCP Dataflow
# =====================================================================
# Note: Make sure you have created your GCS bucket and uploaded
# 'Employers_data.csv' to INPUT_GCS_PATH before running this cell.
# =====================================================================

print(f"Submitting Dataflow Job '{JOB_NAME}' to GCP Project '{PROJECT_ID}'...")
print(f"Reading input from: {INPUT_GCS_PATH}")
print(f"Writing output to:  {OUTPUT_GCS_PATH}")

try:
    with beam.Pipeline(options=dataflow_options) as p:
        (
            p
            | "ReadGCSInput" >> beam.io.ReadFromText(INPUT_GCS_PATH)
            | "ParseAndClean" >> beam.ParDo(ParseAndCleanEmployeeDoFn())
            | "ExtractDeptSalary" >> beam.ParDo(ExtractDepartmentSalaryDoFn())
            | "ComputeAvgSalary" >> combiners.Mean.PerKey()
            | "FormatResults" >> beam.ParDo(FormatSummaryStatsDoFn())
            | "WriteGCSOutput" >> beam.io.WriteToText(
                file_path_prefix=OUTPUT_GCS_PATH,
                file_name_suffix=".csv",
                header="Department,Average_Salary"
            )
        )
    print("\nDataflow job finished or submitted! Visit Google Cloud Console -> Dataflow to view job progress.")
except Exception as err:
    print(f"\nJob execution status / error: {err}")
    print("Verify your GCP project ID, bucket permissions, and GCS input path.")

INFO:root:Runner defaulting to pickling library: cloudpickle.


Submitting Dataflow Job 'employee-salary-dataflow-job' to GCP Project 'debaranjan-practice'...
Reading input from: gs://my-firsct-bucket-debproj/input/Employers_data.csv
Writing output to:  gs://my-firsct-bucket-debproj/output/department_salary_summary


INFO:apache_beam.io.iobase:*** WriteImpl min_shards undef so it's 1, and we write per Bundle
INFO:apache_beam.runners.dataflow.dataflow_runner:Pipeline has additional dependencies to be installed in SDK worker container, consider using the SDK container image pre-building workflow to avoid repetitive installations. Learn more on https://cloud.google.com/dataflow/docs/guides/using-custom-containers#prebuild
INFO:apache_beam.runners.dataflow.internal.apiclient:Starting GCS upload to gs://my-firsct-bucket-debproj/staging/employee-salary-dataflow-job.1790325608.562915/pickled_main_session...
INFO:apache_beam.runners.dataflow.internal.apiclient:Completed GCS upload to gs://my-firsct-bucket-debproj/staging/employee-salary-dataflow-job.1790325608.562915/pickled_main_session in 0 seconds.
INFO:apache_beam.runners.dataflow.internal.apiclient:Starting GCS upload to gs://my-firsct-bucket-debproj/staging/employee-salary-dataflow-job.1790325608.562915/submission_environment_dependencies.txt...
INFO


Dataflow job finished or submitted! Visit Google Cloud Console -> Dataflow to view job progress.


In [16]:
# Cell 9: Verify and Inspect Output Files from Google Cloud Storage
# (Uses user ADC credentials directly)

def inspect_gcs_results(bucket_name: str, prefix: str):
    """Lists and prints the contents of generated output files in GCS."""
    try:
        client = storage.Client(project=PROJECT_ID)
        bucket = client.bucket(bucket_name)
        blobs = list(bucket.list_blobs(prefix=prefix))
        
        if not blobs:
            print(f"No output files found under prefix: '{prefix}'")
            return
            
        print(f"Found {len(blobs)} output partition file(s) in GCS:")
        for blob in blobs:
            print(f"\nBlob: gs://{bucket_name}/{blob.name}")
            content = blob.download_as_text()
            print(content[:500])
    except Exception as err:
        print(f"GCS inspection error: {err}")

# Uncomment the line below after your Dataflow job completes successfully:
inspect_gcs_results(BUCKET_NAME, "output/department_salary_summary")

Found 1 output partition file(s) in GCS:

Blob: gs://my-firsct-bucket-debproj/output/department_salary_summary-00000-of-00001.csv
Department,Average_Salary
Finance,130376.18
Engineering,90680.33
Marketing,101734.57
Product,116676.33
HR,126400.60
Sales,127309.77

